# 肝脏 3D 分割教学教程

本 notebook 以中文分步讲解项目全流程，建议从上到下依次执行每个单元格。

> **免责声明**：本教程为教学演示，使用 **合成的小体积 (32×32×32) 椭球数据** 而非真实 Task03_Liver 数据，确保在无数据、无 GPU 的环境下也能端到端跑通。参数与流程与真实训练一致但规模极小，真实训练请使用 `scripts/train.py` 等命令行脚本。

## 1. 项目目标与背景

**任务**：基于腹部 CT 体数据，进行肝脏与肿瘤的 **3D 体素级分割**。

**两阶段策略**（项目核心设计）：
1. **Phase 1（肝脏分割）**：二分类，背景 vs 肝脏（`num_classes=2`），先定位整个肝脏；
2. **Phase 2（肿瘤分割）**：在 Phase 1 肝脏区域内做三分类，背景 vs 肝脏 vs 肿瘤（`num_classes=3`），细分肿瘤。

**为何两阶段**：肿瘤体积极小、类别极不平衡，直接三分类难以收敛。先粗定位肝脏、再在肝区内细分肿瘤，能显著缓解不平衡问题，提升小目标分割精度。

## 2. 环境与导入

本项目核心建立在 **MONAI**（Medical Open Network for AI）之上：MONAI 是面向医学影像的 PyTorch 上层框架，提供加载、预处理变换、3D 网络、滑动窗口推理、指标等医学影像专用组件，让我们专注于流程而非手写 3D 卷积细节。

In [ ]:
# 导入深度学习与医学影像核心库，并打印版本以便复现
import sys, os
import numpy as np
import torch
import monai

# 将项目根目录加入 sys.path，使 `import src.*` 在 notebook 中可用
sys.path.insert(0, os.path.abspath('..'))

print('Python :', sys.version.split()[0])
print('PyTorch:', torch.__version__)
print('MONAI  :', monai.__version__)
print('NumPy  :', np.__version__)
print('CUDA 可用:', torch.cuda.is_available())

## 3. 数据加载与可视化

为不依赖真实数据，我们 **合成一个 (32,32,32) 的小体积**：用一个椭球模拟肝脏（类别 1），在椭球内部再嵌套一个小椭球模拟肿瘤（类别 2），背景为 0。同时给图像加上随机的“CT 灰度”强度，模拟 HU 值。

**轴向切片（axial slice）**：3D 体沿身体头-脚方向逐层截取的 2D 横断面图像，是放射科阅片最常用的视图，本节取中间一层展示。

In [ ]:
# 合成 (32,32,32) 椭球样本：外层=肝脏(1)，内嵌小椭球=肿瘤(2)
def make_ellipsoid(shape, center, radii):
    """返回布尔体素数组：在椭球内部为 True。"""
    zz, yy, xx = np.indices(shape)  # (D,H,W)
    cz, cy, cx = center
    rz, ry, rx = radii
    return ((zz - cz) / rz) ** 2 + ((yy - cy) / ry) ** 2 + ((xx - cx) / rx) ** 2 <= 1.0

SHAPE = (32, 32, 32)
liver_mask = make_ellipsoid(SHAPE, center=(16, 16, 16), radii=(10, 12, 12)).astype(np.int8)
tumor_mask = make_ellipsoid(SHAPE, center=(20, 12, 20), radii=(2, 3, 3)).astype(np.int8)

# 组合成多类标签：肿瘤(2) 覆盖肝脏(1)
label = liver_mask.copy()
label[tumor_mask == 1] = 2

# 合成“CT 灰度”：背景 ~ -100，肝脏 ~ 60，肿瘤 ~ 80，加少量噪声
rng = np.random.default_rng(0)
image = np.full(SHAPE, -100.0, dtype=np.float32)
image[label == 1] = 60.0
image[label == 2] = 80.0
image += rng.normal(0, 5, SHAPE).astype(np.float32)

print('合成图像 shape:', image.shape, 'dtype:', image.dtype)
print('合成标签取值:', np.unique(label))

# 可视化第 16 层 axial 切片
import matplotlib.pyplot as plt
z = 16
fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(image[z], cmap='gray'); axes[0].set_title(f'CT axial z={z}'); axes[0].axis('off')
axes[1].imshow(label[z], cmap='nipy_spectral'); axes[1].set_title('标签 (0背景/1肝/2瘤)'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 4. 数据预处理变换演示

项目用 `src.data.get_train_transforms` 构建一轮 MONAI 字典变换流水线（对 `{'image':..., 'label':...}` 同时操作）。其顺序为：`LoadImaged → EnsureChannelFirstD → OrientationD → SpacingD → ScaleIntensityRangeD → RandCropFgD → RandFlipd → EnsureTyped`。

这里为演示直观，直接用 **MONAI 单个变换**逐一展示其中三步的作用（合成数据跳过文件加载与方向统一）：
- `Spacing`：重采样到统一体素间距，消除不同扫描分辨率差异；
- `ScaleIntensityRange`：把 CT HU 窗线性归一化到 [0,1]，聚焦软组织；
- `RandSpatialCrop`：随机裁剪固定大小子块，提升数据效率与前景区采样。

In [ ]:
# 演示三个核心 MONAI 变换：Spacing / ScaleIntensityRange / RandSpatialCrop
from monai.transforms import Spacingd, ScaleIntensityRangeD, RandSpatialCropd, EnsureChannelFirstd, ToTensord

# MONAI 变换要求通道在前：(C,D,H,W)
data = {'image': image[None].copy(), 'label': label[None].astype(np.float32).copy()}
data = EnsureChannelFirstd(keys=['image', 'label'])(data)  # 此处已是 (1,32,32,32)，幂等演示

# 1) Spacing：把体素间距重采样到 (1,1,1)mm。合成数据 spacing 视为已统一，仅演示调用形式
spacing_t = Spacingd(keys=['image', 'label'], pixdim=(1, 1, 1), mode=('bilinear', 'nearest'))
data = spacing_t(data)

# 2) ScaleIntensityRange：把 HU 窗 [-100, 80] 线性映射到 [0, 1]
scale_t = ScaleIntensityRangeD(keys=['image'], a_min=-100, a_max=80, b_min=0.0, b_max=1.0, clip=True)
data = scale_t(data)

# 3) RandSpatialCrop：随机裁剪 (16,16,16) 子块（合成演示用小尺寸）
crop_t = RandSpatialCropd(keys=['image', 'label'], roi_size=(16, 16, 16), random_size=False)
data_cropped = crop_t(data)

print('变换后图像 shape:', data['image'].shape, '强度范围:', float(data['image'].min()), '~', float(data['image'].max()))
print('裁剪后 shape  :', data_cropped['image'].shape)

## 5. 模型结构

项目用 `src.models.build_model(config)` 构建 **MONAI 3D U-Net**。U-Net 形如字母 U：编码器（左下行）逐层下采样提取从低级到高级特征；瓶颈（底部）做全局理解；解码器（右上行）逐层上采样恢复分辨率；**跳跃连接**把编码器精细特征拼到解码器，让网络“既懂全局又画得准边缘”。

此处用一个小 config 演示构建与参数量统计。

In [ ]:
from src.models import build_model

# 极小 config 演示：通道数与层数都降到很小，便于 CPU 快速构建查看
config = {
    'model': {
        'in_channels': 1,          # 单模态 CT
        'out_channels': 3,         # Phase2：背景/肝/瘤
        'channels': (8, 16, 32),   # 三层特征通道（小规模演示）
        'strides': (2, 2),         # 两次下采样，长度 = len(channels)-1
        'num_res_units': 1,        # 每块残差单元数
        'norm': 'BATCH',           # 批归一化
    }
}
model = build_model(config)

# 打印结构与参数量
n_params = sum(p.numel() for p in model.parameters())
print(model)
print(f'参数量 = {n_params:,} ({n_params / 1e6:.2f} M)')

## 6. 训练一步演示

用 `src.trainer.train_one_epoch` 跑一个 epoch。为不依赖真实数据，我们用合成椭球构造一个极小的 batch（`DataLoader`），并配置 tiny loss / optimizer。

> **说明**：这只是教学演示，验证“前向→loss→反向→更新”闭环是否通。真实训练请用 `scripts/train.py` 跑完整流程（大体积、大 batch、多 epoch、AMP、checkpoint）。

In [ ]:
from torch.utils.data import DataLoader, Dataset
from src.trainer import train_one_epoch
from monai.losses import DiceCELoss

# 用上一步的合成像/标签构造 4 个样本的迷你数据集
class SyntheticSet(Dataset):
    def __init__(self, image, label, n=4):
        self.image = image[None].astype(np.float32)  # (1,D,H,W)
        self.label = label[None].astype(np.int64)
        self.n = n
    def __len__(self): return self.n
    def __getitem__(self, i):
        return {'image': torch.from_numpy(self.image.copy()),
                'label': torch.from_numpy(self.label.copy())}

loader = DataLoader(SyntheticSet(image, label, n=4), batch_size=2, shuffle=True)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = model.to(device)

# build_loss 需 config['loss']，这里直接用 MONAI DiceCELoss 演示（to_onehot_y=True 适配类别 id 标签）
loss_fn = DiceCELoss(to_onehot_y=True, softmax=True)
optimizer = torch.optim.Adam(model.parameters(), lr=1e-3)

avg_loss = train_one_epoch(model, loader, loss_fn, optimizer, device, amp=False)
print(f'一个 epoch 平均 loss = {avg_loss:.4f}（教学演示，数值无实际意义）')

## 7. 推理与 argmax 解码

推理阶段网络输出形如 `(C,D,H,W)` 的类别得分，需经 **softmax + argmax** 解码成每个体素的类别 id。对于大体积，项目用 **滑动窗口推理（SlidingWindowInferer）** 分块预测再拼回，避免显存溢出。

这里用合成小体积直接演示 argmax 与滑动窗口两种方式的一致性。

In [ ]:
from monai.inferers import SlidingWindowInferer

model.eval()
with torch.no_grad():
    x = torch.from_numpy(image[None][None].astype(np.float32)).to(device)  # (1,1,D,H,W)
    logits = model(x)                                    # (1,C,D,H,W) 原始得分
    pred_direct = logits.argmax(dim=1).squeeze().cpu().numpy()  # argmax 解码到类别 id

    # 滑动窗口：roi=(16,16,16) 演示（真实场景用 96³）
    inferer = SlidingWindowInferer(roi_size=(16, 16, 16), sw_batch_size=1, overlap=0.25)
    logits_sw = inferer(x, model)
    pred_sw = logits_sw.argmax(dim=1).squeeze().cpu().numpy()

print('argmax 解码类别取值:', np.unique(pred_direct))
print('两方式结果是否一致:', np.array_equal(pred_direct, pred_sw))

fig, axes = plt.subplots(1, 2, figsize=(8, 4))
axes[0].imshow(pred_direct[16], cmap='nipy_spectral'); axes[0].set_title('直接 argmax z=16'); axes[0].axis('off')
axes[1].imshow(pred_sw[16], cmap='nipy_spectral'); axes[1].set_title('滑动窗口 + argmax z=16'); axes[1].axis('off')
plt.tight_layout(); plt.show()

## 8. 指标解读

项目用 `src.trainer.compute_metrics(pred_seg, label_seg, num_classes)` 计算单样本 3D 指标。对每个前景类别 c：

- **Dice** = 2·TP / (2·TP + FP + FN)：与 F1 等价，衡量重叠度；
- **IoU** = TP / (TP + FP + FN)：交并比，比 Dice 更严；
- **Precision** = TP / (TP + FP)：预测为 c 的体素里有多少真是 c；
- **Recall** = TP / (TP + FN)：真值为 c 的体素里有多少被找出。

**空真值处理**：若某类在真值中完全不存在（如某样本无肿瘤），则该类四项指标记为 `NaN`，避免用 0 分把“没有该类”误判为“分割差”，aggregate 时再忽略空真值样本平均，更公平。

In [ ]:
import math
from src.trainer import compute_metrics

# 用“完全等于真值”的预测做正向演示
pred_perfect = label.copy()
metrics_perfect = compute_metrics(pred_perfect, label, num_classes=3)
print('== 完美预测指标 ==')
for k, v in metrics_perfect.items():
    print(f'  {k}: {v:.4f}' if not math.isnan(v) else f'  {k}: NaN')

# 空 tumor 真值演示：把 tumor 全部去掉
label_no_tumor = label.copy(); label_no_tumor[label_no_tumor == 2] = 1
pred_all_liver = np.ones_like(label)  # 预测全为肝脏
metrics_empty = compute_metrics(pred_all_liver, label_no_tumor, num_classes=3)
print('\n== 真值无 tumor 时指标 ==')
for k, v in metrics_empty.items():
    print(f'  {k}: {v:.4f}' if not math.isnan(v) else f'  {k}: NaN（空真值）')

## 9. STL 导出演示

项目提供 `src.utils.export_label_stl(label, label_id, output_path, spacing, smooth)`：先按 `label_id` 取出该类二值掩膜，再用 marching cubes 提取等值面网格并写入 STL 文件，方便在 3D Slicer / MeshLab 等软件中可视化分割结果。

此处导出合成小掩膜中肝脏（label_id=1）的 STL。

In [ ]:
from src.utils import export_label_stl

out_dir = os.path.join('..', 'outputs', 'tutorial')
os.makedirs(out_dir, exist_ok=True)
stl_path = os.path.join(out_dir, 'synthetic_liver.stl')

# 导出合成标签中的肝脏（类别 1）为 STL
export_label_stl(label, label_id=1, output_path=stl_path, spacing=(1.0, 1.0, 1.0), smooth=False)
print('已导出 STL:', stl_path, '存在:', os.path.exists(stl_path))

## 10. 完整流程回顾

本教程以合成小数据走通了全流程，对应项目命令行脚本：

- 数据/预处理逻辑：`src/data.py`；
- 模型构建：`src/models.py`；
- 训练 / 指标 / 推理：`src/trainer.py`、`src/utils.py`；
- 完整训练入口：`scripts/train.py`（多 epoch、checkpoint、AMP、多卡）；
- 完整推理入口：`scripts/infer.py`；
- STL 导出入口：`scripts/export_stl.py`。

**鼓励**：理解本教程后，请用 `scripts/*.py` 在真实 Task03_Liver 数据上跑端到端训练与推理，体验完整规模下的医学影像 3D 分割工作流。